In [ ]:
# useful to autoreload the module without restarting the kernel
%load_ext autoreload
%autoreload 2

In [ ]:
from mppi import InputFiles as I, Calculators as C, Datasets as D
from mppi.Datasets import PostProcessing as PP

# yambo_nl simulations of bulk LiF: delta, sine and frequency mixing

This notebook builds and runs the `yambo_nl` simulations used to extract the equilibrium (non-linear) susceptibilities
of LiF. The analysis of the results (extraction of the $\chi$ and reconstruction of $\delta\chi^{neq}$ for the
comparison with the RT transient absorption) is performed in `NL-Chi_Analysis.ipynb`.

We use the SAVE folder computed by D. Sangalli in

_/work/sangalli/simulations/LiF/yambo/e80_kx6_pbe_sr/kx8_nb100/NoTr_E100_

(kx8 grid, 100 bands, time reversal removed for a field along 100), copied in `kx8_nb100_NoTr_E100/`.

Three kinds of simulations are performed:

- a single delta pulse (linear response);
- a set of monochromatic sine-shaped probes in the 10-20 eV range;
- frequency mixing: the same probes with a sine-shaped pump at 1.55 eV.

All the runs are appended with `skip=True`: if the results are already present in the run folder they are not recomputed.

## RT simulation with yambo_nl in the IP approximation

The parameters are taken from the folder

_/work/sangalli/simulations/LiF/yambo/e80_kx6_pbe_sr/kx8_nb100/NoTr_E100/sh6.0+str0.05_probes/tdberry_

and in particular from `td-ip-ber_3.0as_inv_3-12b_delta1.0fs_deph0.1fs_p_NL`, reducing the bands to 3-6.

In [ ]:
# RunRules for ismhpc
# The product of ntasks_per_node*cpus_per_task is equal to 32. 
# Many mpi are needed for better performances
nodes = 1
ntasks_per_node = 32 #16
cpus_per_task = 1 #2
omp_num_threads = 1 #2
module = '/home/dalessandro/module_script/yambo_module'

ntasks = nodes*ntasks_per_node

rr = C.RunRules(scheduler='slurm',partition='all12h', #'all12h',
                memory='125000',
                nodes=nodes,ntasks_per_node=ntasks_per_node,
                cpus_per_task=cpus_per_task,
                omp_num_threads=omp_num_threads,pre_processing=module)
code = C.YamboCalculator(rr,executable='yambo_nl',activate_BeeOND=True)
#code.global_options()

In [ ]:
run_dir = 'kx8_nb100_NoTr_E100/'

### Linear response within the Berry phase approach (delta pulse)

We compute the linear response of the system using a single delta shaped pulse. The linear $\chi$ extracted from this
run is compared with the one computed from the sine-shaped pulses below.

In [ ]:
NLBands = [3,6]
GfnQP_E = [6.,1.05,1.05]

Field1_Int = 1.e3 
Field1_Dir = [1.,0.,0.]
Field1_kind = 'DELTA'
Field1_Tstart = 0.1 # fs
NLDamping = 0.0 # eV
NLtime, NLstep = 100, 0.05  # fs
IOtime = [1.,5.,0.005] #fs

inp = I.YamboInput(args='yambo_nl -u p',folder=run_dir)
inp.set_array_variables(NLBands=NLBands,GfnQP_E=GfnQP_E)
inp.set_array_variables(units='eV',NLDamping=NLDamping)
inp.set_array_variables(units='kWLm2',Field1_Int=Field1_Int)
inp.set_array_variables(Field1_Dir=Field1_Dir)
inp.set_array_variables(units='fs',Field1_Tstart=Field1_Tstart,NLtime=NLtime,NLstep=NLstep)
inp.set_scalar_variables(Field1_kind=Field1_kind)
inp.set_array_variables(units='fs',IOtime=IOtime)

study = D.Dataset(num_tasks=1,run_dir=run_dir,verbose=True)
study.set_postprocessing_function(PP.yambo_parse_data)
idd = 'lresponse-bands_3-6-delta'
study.append_run(id=idd,input=inp,runner=code,jobname=[idd],skip=True)

In [ ]:
results = study.run()

### Sine-shaped probe

We compute the response of the system to a set of monochromatic sine-shaped pulses (one run for each probe frequency)
with an intensity of 1e3 kW/m$^2$. These runs provide the linear $\chi$ in the probe energy range.

In [ ]:
NLBands = [3,6]
GfnQP_E = [6.,1.05,1.05]
NLEnRange = [10.0,20.0] # energy range of the probe
NLEnSteps = 201 # number of steps for the loop on frequencies

Field1_Int = 1e3
Field1_Dir = [1.,0.,0.]
Field1_kind = 'SIN'
Field1_Freq = -0.1 # the range is used to set th freq of the pulse
Field1_Tstart = 0.1 # fs
NLDamping = 0.1 # eV
NLtime, NLstep = 200, 0.01  # fs
IOtime = [1.,5.,0.005] #fs

inp = I.YamboInput(args='yambo_nl -u n',folder=run_dir)
inp.set_array_variables(NLBands=NLBands,GfnQP_E=GfnQP_E)
inp.set_array_variables(units='eV',NLDamping=NLDamping,Field1_Freq=Field1_Freq,NLEnRange=NLEnRange)
inp.set_array_variables(units='kWLm2',Field1_Int=Field1_Int)
inp.set_array_variables(Field1_Dir=Field1_Dir,NLEnSteps=NLEnSteps)
inp.set_array_variables(units='fs',Field1_Tstart=Field1_Tstart,NLtime=NLtime,NLstep=NLstep)
inp.set_scalar_variables(Field1_kind=Field1_kind)
inp.set_array_variables(units='fs',IOtime=IOtime)

study = D.Dataset(num_tasks=1,run_dir=run_dir,verbose=True)
study.set_postprocessing_function(PP.yambo_parse_data)
idd = 'pulse-E1_1e3-nlenrange_%s-%s-nlensteps_%s-bands_3-6-damp_0.1-sin'%(NLEnRange[0],NLEnRange[1],NLEnSteps)
study.append_run(id=idd,input=inp,runner=code,jobname=[idd],skip=True) #,reformat=False

In [ ]:
results = study.run()

### Frequency mixing: probe and pump

The same set of probes is combined with a sine-shaped pump at 1.55 eV with an intensity of 1e6 kW/m$^2$. These runs
provide the mixing susceptibilities $\chi_{(n,m)}$ (n times the probe and m times the pump frequency).

In [ ]:
NLBands = [3,6]
GfnQP_E = [6.,1.05,1.05]
NLEnRange = [10.0,20.0] # energy range of the probe
NLEnSteps = 201 

Field1_Freq, Field2_Freq = -0.1, 1.55 # eV Energy of the pump pulse
Field1_Int, Field2_Int = 1.e3, 1.e6 # kW/m^2 (probe, pump)
Field1_Dir, Field2_Dir = [1.,0.,0.], [1.,0.,0.]
Field1_kind, Field2_kind = 'SIN', 'SIN'
Field1_Tstart, Field2_Tstart = 0.1, 0.1 # fs
NLDamping = 0.1 # eV
NLtime, NLstep = 200, 0.01  # fs
IOtime = [1.,5.,0.005] #fs

inp = I.YamboInput(args='yambo_nl -u n',folder=run_dir)
inp.set_array_variables(NLBands=NLBands,GfnQP_E=GfnQP_E,NLEnSteps=NLEnSteps)
inp.set_array_variables(units='eV',NLEnRange=NLEnRange,Field1_Freq=Field1_Freq,Field2_Freq=Field2_Freq,NLDamping=NLDamping)
inp.set_array_variables(units='kWLm2',Field1_Int=Field1_Int,Field2_Int=Field2_Int)
inp.set_array_variables(Field1_Dir=Field1_Dir,Field2_Dir=Field2_Dir)
inp.set_array_variables(units='fs',Field1_Tstart=Field1_Tstart,Field2_Tstart=Field2_Tstart,NLtime=NLtime,NLstep=NLstep)
inp.set_scalar_variables(Field1_kind=Field1_kind,Field2_kind=Field2_kind)
inp.set_array_variables(units='fs',IOtime=IOtime)
inp.set_scalar_variables(NL_ROLEs="w.k",NL_CPU='4.8')

study = D.Dataset(num_tasks=1,run_dir=run_dir,verbose=True)
study.set_postprocessing_function(PP.yambo_parse_data)
idd = 'Pp-Ep_1e3-EP_1e6-PE_1.55-nlenrange_%s-%s-nlensteps_%s-bands_3-6-damp_0.1-sin'%(NLEnRange[0],NLEnRange[1],NLEnSteps)
study.append_run(id=idd,input=inp,runner=code,jobname=[idd],skip=True)

In [ ]:
results = study.run()